# Import library and dataset

In [ ]:
#Initial setup
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display

import pandas as pd
import os
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from decimal import Decimal, ROUND_HALF_UP
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LassoCV
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.linear_model import LogisticRegression
import urllib.request
import joblib

In [ ]:
fileDir = "./Data/"
fileName = "Eigengene_ME.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
ME_ML = temp
display(ME_ML)

In [ ]:
fileDir = "./Data/"
fileName = "WGCNA_module_assignment.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
module_tbl = temp
display(module_tbl)

# random forest

In [ ]:
ME_features = [
    "MEblue",
    "MEbrown",
    "MEgreen",
    # "MEgrey",
    "MEred",
    "MEturquoise",
    "MEyellow"
]

In [ ]:
X = ME_ML[ME_features]
y = ME_ML["CaCo_Status"]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
aucs = []
fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
    param_grid = {
        'n_estimators': [100, 300, 500],
        'max_depth': [None, 3, 5, 7, 10],
        # 'min_samples_split': [2, 3, 4],
        'min_samples_leaf': [1, 2, 4],
        'max_features': [None, 'sqrt'],
        'class_weight': ['balanced', None]
    }
    
    model = RandomForestClassifier(
        criterion='gini',
        random_state=123,
        # class_weight='balanced',
        n_jobs=-1 
    )
    
    cv_gen = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    
    modelcv = GridSearchCV(
        model, 
        param_grid=param_grid,
        scoring='roc_auc',
        n_jobs=-1, 
        refit=True, 
        cv=cv_gen,
        verbose=0, 
        pre_dispatch='2*n_jobs', 
        error_score=np.nan, 
        return_train_score=False
    )
    
    # 訓練の実行
    modelcv.fit(X_train, y_train)
    
    print('Best score: ', modelcv.best_score_)
    print('Best hyperparameters: ', modelcv.best_params_)
    dataframe = pd.DataFrame(modelcv.cv_results_).sort_values('mean_test_score', ascending=False)
    display(dataframe)
    
    model = modelcv.best_estimator_
    y_prob = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    print('AUC: ', score)
    
    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {i+1}", ax=ax
    )
    
print(f"5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs)}")
plt.show()



importance_df = pd.DataFrame({
    "Protein": X.columns,
    "Importance": model.feature_importances_
})
importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
)
print(importance_df.head(20))



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    square=True
)

plt.title("Correlation among WGCNA eigengenes")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
aucs = []
fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
    param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'],
    'solver': ['lbfgs'],
    'class_weight': [None, 'balanced']
}
    
    model = LogisticRegression(
    max_iter=5000
)
    
    
    modelcv = GridSearchCV(
        model, 
        param_grid=param_grid,
        scoring='roc_auc',
        n_jobs=-1, 
        refit=True, 
        cv=cv_gen,
        verbose=0, 
        pre_dispatch='2*n_jobs', 
        error_score=np.nan, 
        return_train_score=False
    )
    
    # 訓練の実行
    modelcv.fit(X_train, y_train)
    
    print('Best score: ', modelcv.best_score_)
    print('Best hyperparameters: ', modelcv.best_params_)
    dataframe = pd.DataFrame(modelcv.cv_results_).sort_values('mean_test_score', ascending=False)
    display(dataframe)
    
    model = modelcv.best_estimator_
    y_prob = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    print('AUC: ', score)
    
    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {i+1}", ax=ax
    )
    
print(f"5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs)}")
plt.show()


importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

importance_df["Abs_Coefficient"] = importance_df["Coefficient"].abs()

importance_df = importance_df.sort_values(
    "Abs_Coefficient",
    ascending=False
)

print(importance_df)

## MEturquoise

In [ ]:
X = ME_ML[["MEturquoise"]]
y = ME_ML["CaCo_Status"]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
aucs = []
fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
    param_grid = {
        'n_estimators': [100, 300, 500],
        'max_depth': [None, 3, 5, 7, 10],
        # 'min_samples_split': [2, 3, 4],
        'min_samples_leaf': [1, 2, 4],
        'max_features': [None, 'sqrt'],
        'class_weight': ['balanced', None]
    }
    
    model = RandomForestClassifier(
        criterion='gini',
        random_state=123,
        # class_weight='balanced',
        n_jobs=-1 
    )
    
    cv_gen = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    
    modelcv = GridSearchCV(
        model, 
        param_grid=param_grid,
        scoring='roc_auc',
        n_jobs=-1, 
        refit=True, 
        cv=cv_gen,
        verbose=0, 
        pre_dispatch='2*n_jobs', 
        error_score=np.nan, 
        return_train_score=False
    )
    
    # 訓練の実行
    modelcv.fit(X_train, y_train)
    
    print('Best score: ', modelcv.best_score_)
    print('Best hyperparameters: ', modelcv.best_params_)
    dataframe = pd.DataFrame(modelcv.cv_results_).sort_values('mean_test_score', ascending=False)
    display(dataframe)
    
    model = modelcv.best_estimator_
    y_prob = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    print('AUC: ', score)
    
    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {i+1}", ax=ax
    )
    
print(f"5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs)}")
plt.show()



importance_df = pd.DataFrame({
    "Protein": X.columns,
    "Importance": model.feature_importances_
})
importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
)
print(importance_df.head(20))



## MEblue

In [ ]:
X = ME_ML[["MEblue"]]
y = ME_ML["CaCo_Status"]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
aucs = []
fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
    param_grid = {
        'n_estimators': [100, 300, 500],
        'max_depth': [None, 3, 5, 7, 10],
        # 'min_samples_split': [2, 3, 4],
        'min_samples_leaf': [1, 2, 4],
        'max_features': [None, 'sqrt'],
        'class_weight': ['balanced', None]
    }
    
    model = RandomForestClassifier(
        criterion='gini',
        random_state=123,
        # class_weight='balanced',
        n_jobs=-1 
    )
    
    cv_gen = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    
    modelcv = GridSearchCV(
        model, 
        param_grid=param_grid,
        scoring='roc_auc',
        n_jobs=-1, 
        refit=True, 
        cv=cv_gen,
        verbose=0, 
        pre_dispatch='2*n_jobs', 
        error_score=np.nan, 
        return_train_score=False
    )
    
    # 訓練の実行
    modelcv.fit(X_train, y_train)
    
    print('Best score: ', modelcv.best_score_)
    print('Best hyperparameters: ', modelcv.best_params_)
    dataframe = pd.DataFrame(modelcv.cv_results_).sort_values('mean_test_score', ascending=False)
    display(dataframe)
    
    model = modelcv.best_estimator_
    y_prob = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    print('AUC: ', score)
    
    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {i+1}", ax=ax
    )
    
print(f"5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs)}")
plt.show()



importance_df = pd.DataFrame({
    "Protein": X.columns,
    "Importance": model.feature_importances_
})
importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
)
print(importance_df.head(20))



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, RocCurveDisplay
from sklearn.inspection import permutation_importance

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

aucs = []
fig, ax = plt.subplots()

best_models = []

for i, (train_index, test_index) in enumerate(skf.split(X, y)):

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # 標準化 + ロジスティック回帰
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(
            solver="saga",
            max_iter=10000,
            random_state=123
        ))
    ])

    param_grid = [
        {
            "logreg__penalty": ["l1", "l2"],
            "logreg__C": [0.001, 0.01, 0.1, 1, 10, 100],
            "logreg__class_weight": [None, "balanced"]
        },
        {
            "logreg__penalty": ["elasticnet"],
            "logreg__C": [0.001, 0.01, 0.1, 1, 10, 100],
            "logreg__l1_ratio": [0.25, 0.5, 0.75],
            "logreg__class_weight": [None, "balanced"]
        }
    ]

    cv_gen = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=123
    )

    modelcv = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring="roc_auc",
        n_jobs=-1,
        refit=True,
        cv=cv_gen,
        verbose=0,
        error_score=np.nan,
        return_train_score=False
    )

    # 内側のCVでパラメータ探索
    modelcv.fit(X_train, y_train)

    print(f"\nFold {i+1}")
    print("Best inner CV AUC:", modelcv.best_score_)
    print("Best hyperparameters:", modelcv.best_params_)

    dataframe = (
        pd.DataFrame(modelcv.cv_results_)
        .sort_values("mean_test_score", ascending=False)
    )
    display(dataframe.head(10))

    # 外側のテストfoldで評価
    model = modelcv.best_estimator_
    best_models.append(model)

    y_prob = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    print("Test AUC:", score)

    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {i+1}",
        ax=ax
    )

print(f"\n5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")

plt.legend()
plt.show()

## MEyellow

In [ ]:
X = ME_ML[["MEyellow"]]
y = ME_ML["CaCo_Status"]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
aucs = []
fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
    param_grid = {
        'n_estimators': [100, 200, 300, 500],
        'max_depth': [None, 20, 30,40],
        'min_samples_split': [2, 3, 4],
        'min_samples_leaf': [1, 2, 3]
    }
    
    model = RandomForestClassifier(
        criterion='gini',
        random_state=123,
        class_weight='balanced',
        n_jobs=-1 
    )
    
    cv_gen = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    
    modelcv = GridSearchCV(
        model, 
        param_grid=param_grid,
        scoring='roc_auc',
        n_jobs=-1, 
        refit=True, 
        cv=cv_gen,
        verbose=0, 
        pre_dispatch='2*n_jobs', 
        error_score=np.nan, 
        return_train_score=False
    )
    
    # 訓練の実行
    modelcv.fit(X_train, y_train)
    
    print('Best score: ', modelcv.best_score_)
    print('Best hyperparameters: ', modelcv.best_params_)
    dataframe = pd.DataFrame(modelcv.cv_results_).sort_values('mean_test_score', ascending=False)
    display(dataframe)
    
    model = modelcv.best_estimator_
    y_prob = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    print('AUC: ', score)
    
    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {i+1}", ax=ax
    )
    
print(f"5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs)}")
plt.show()



importance_df = pd.DataFrame({
    "Protein": X.columns,
    "Importance": model.feature_importances_
})
importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
)
print(importance_df.head(20))



## MEblue + MEturquoise

In [ ]:
X = ME_ML[["MEblue", "MEturquoise"]]
y = ME_ML["CaCo_Status"]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
aucs = []
fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
    param_grid = {
        'n_estimators': [100, 300, 500],
        'max_depth': [None, 3, 5, 7, 10],
        # 'min_samples_split': [2, 3, 4],
        'min_samples_leaf': [1, 2, 4],
        'max_features': [None, 'sqrt'],
        'class_weight': ['balanced', None]
    }
    
    model = RandomForestClassifier(
        criterion='gini',
        random_state=123,
        # class_weight='balanced',
        n_jobs=-1 
    )
    
    cv_gen = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    
    modelcv = GridSearchCV(
        model, 
        param_grid=param_grid,
        scoring='roc_auc',
        n_jobs=-1, 
        refit=True, 
        cv=cv_gen,
        verbose=0, 
        pre_dispatch='2*n_jobs', 
        error_score=np.nan, 
        return_train_score=False
    )
    
    # 訓練の実行
    modelcv.fit(X_train, y_train)
    
    print('Best score: ', modelcv.best_score_)
    print('Best hyperparameters: ', modelcv.best_params_)
    dataframe = pd.DataFrame(modelcv.cv_results_).sort_values('mean_test_score', ascending=False)
    display(dataframe)
    
    model = modelcv.best_estimator_
    y_prob = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    print('AUC: ', score)
    
    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {i+1}", ax=ax
    )
    
print(f"5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs)}")
plt.show()



importance_df = pd.DataFrame({
    "Protein": X.columns,
    "Importance": model.feature_importances_
})
importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
)
print(importance_df.head(20))



# ProgressをYに

In [ ]:
display(ME_ML)

In [ ]:
X = ME_ML[ME_features]
y = ME_ML["progress"].astype(str)

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

print(y.unique())
print(y.value_counts())

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
aucs = []
fig, ax = plt.subplots()


for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
    param_grid = {
        'n_estimators': [100, 200, 300, 500],
        'max_depth': [None, 20, 30,40],
        'min_samples_split': [2, 3, 4],
        'min_samples_leaf': [1, 2, 3],
        "class_weight": ["balanced", None]
    }
    
    model = RandomForestClassifier(
        criterion='gini',
        random_state=123,
        n_jobs=-1 
    )
    
    cv_gen = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    
    modelcv = GridSearchCV(
        model, 
        param_grid=param_grid,
        scoring='roc_auc',
        n_jobs=-1, 
        refit=True, 
        cv=cv_gen,
        verbose=0, 
        pre_dispatch='2*n_jobs', 
        error_score=np.nan, 
        return_train_score=False
    )
    
    # 訓練の実行
    modelcv.fit(X_train, y_train)
    
    print('Best score: ', modelcv.best_score_)
    print('Best hyperparameters: ', modelcv.best_params_)
    dataframe = pd.DataFrame(modelcv.cv_results_).sort_values('mean_test_score', ascending=False)
    display(dataframe)
    
    model = modelcv.best_estimator_
    y_prob = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    print('AUC: ', score)
    
    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {i+1}", ax=ax
    )
    
print(f"5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs)}")
plt.show()



importance_df = pd.DataFrame({
    "Protein": X.columns,
    "Importance": model.feature_importances_
})
importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
)
print(importance_df.head(20))

